Macro assemblers are cool
A constraint based one would be cool.
The algebra of assembly
FASM
Typed Assembly
Be Micro


In [12]:
from dataclasses import dataclass
@dataclass(frozen=True, slots=True)
class Insn:
    op : str
    args : tuple[str,...]

    def __str__(self):
        return f"{self.op} {", ".join(self.args)}"
@dataclass
class Label:
    name : str
    def __str__(self):
        return f"{self.name}:"
r0,r1,r2,r3,r4,r5,r6 = "r0 r1 r2 r3 r4 r5 r6".split()
def mov(x,y): return Insn("mv", (x,y))
prog = [
    Label("load"),
    mov(r0,r1),
    mov(r2,r3)
]
import subprocess
def asm(insns):
    with open("/tmp/prog.s", "w") as f:
        f.write("\n".join(str(i) for i in insns))
    subprocess.run(["riscv64-unknown-elf-as", "/tmp/prog.s", "-o", "/tmp/prog.o"], check=True)
    return subprocess.run(["objdump", "-d", "/tmp/prog.o"], check=True).stdout.decode("utf-8")

asm(prog)


/tmp/prog.s: Assembler messages:
/tmp/prog.s: Warning: end of file not at end of a line; newline inserted
/tmp/prog.s:2: Error: illegal operands `mv r0,r1'
/tmp/prog.s:3: Error: illegal operands `mv r2,r3'


CalledProcessError: Command '['riscv64-unknown-elf-as', '/tmp/prog.s', '-o', '/tmp/prog.o']' returned non-zero exit status 1.

# wasm



In [4]:
from wasmtime import Store, Module, Instance, Func, FuncType
help(Module)

Help on class Module in module wasmtime._module:

class Module(wasmtime._managed.Managed)
 |  Module(engine: wasmtime._engine.Engine, wasm: Union[str, bytes])
 |
 |  Method resolution order:
 |      Module
 |      wasmtime._managed.Managed
 |      typing.Generic
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, engine: wasmtime._engine.Engine, wasm: Union[str, bytes])
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  serialize(self) -> bytearray
 |      Serializes this module to a binary representation.
 |
 |      This method will serialize this module to an in-memory byte array which
 |      can be cached and later passed to `Module.deserialize` to recreate this
 |      module.
 |
 |  ----------------------------------------------------------------------
 |  Class methods defined here:
 |
 |  deserialize(engine: wasmtime._engine.Engine, encoded: Union[bytes, bytearray]) -> 'Module'
 |      Deserializes bytes previously created by `M

In [6]:
from wasmtime import Store, Module, Instance, Func, FuncType


prog = """
(module
  (func $hello (import "" "hello"))
  (func (export "run") (call $hello))
)
"""

store = Store()
module = Module(store.engine,prog) #Module.from_file(store.engine, './examples/hello.wat')
def say_hello():
    print("Hello from Python!")

hello = Func(store, FuncType([], []), say_hello)

# And with all that we can instantiate our module and call the export!
instance = Instance(store, module, [hello])
instance.exports(store)["run"](store)

Hello from Python!


In [ ]:
progmyadd = """
(module
 (func $myadd (param i32 i32) (result i32)
    local.get 0
    local.get 1
    i32.add
)
)
"""

In [ ]:
@dataclass
class Insn:
    